# Notebook 4 — Lightweight Pretrained Models: MobileNetV2 & EfficientNet-B0

## Story
All previous notebooks trained from scratch.  The fundamental limitation:
**our dataset is too small to learn rich visual representations from random
initialisation**.

**Transfer learning** lets us start from models already trained on ImageNet
(1.2 M images, 1000 classes).  Those backbones have already learned to detect
edges, textures, shapes and object parts — exactly the features we need.

We use two *lightweight* architectures designed for efficiency:

| Model | Params | Key Innovation |
|---|---|---|
| **MobileNetV2** | ~3.4 M | Depthwise-separable convolutions; very fast. |
| **EfficientNet-B0** | ~5.3 M | Compound scaling of depth/width/resolution. |

Strategy: freeze most backbone weights, fine-tune only the last block + new head.

**Labels (12):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, "../")
sys.path.insert(0, "../experiments")

import torch
import torch.nn as nn
from pathlib import Path
from torchvision import models as tv_models

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, split_dataset, subsample_subset,
    get_train_transform, get_eval_transform, build_dataloaders,
    run_baselines, print_model_info,
    train_model, save_checkpoint, load_checkpoint, plot_training_history,
    plot_multi_arch_histories,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, show_saliency_examples,
    compute_multilabel_metrics, NUM_LABELS, METRIC_KEYS,
)

SEED   = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Config

In [ ]:
BASE_DATA_DIR   = "../data/aggregated"
IMAGE_SIZE      = 128
BATCH_SIZE      = 128
SPLIT           = [0.8, 0.1, 0.1]
USE_SUBSET      = False
SUBSET_FRACTION = 0.1
CHECKPOINT_DIR  = Path("../checkpoints")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")

## 3. Data Loading & Loaders

In [ ]:
full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)

if USE_SUBSET:
    train_raw = subsample_subset(train_raw, SUBSET_FRACTION, SEED)
    val_raw   = subsample_subset(val_raw,   SUBSET_FRACTION, SEED + 1)
    test_raw  = subsample_subset(test_raw,  SUBSET_FRACTION, SEED + 2)

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform  = get_eval_transform(IMAGE_SIZE)
train_loader, val_loader, test_loader = build_dataloaders(
    train_raw, val_raw, test_raw, train_transform, eval_transform, BATCH_SIZE
)
print(f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")

## 4. Why Transfer Learning? — A Quick Intuition

ImageNet-pretrained features form a hierarchy:
- **Early layers** detect edges and colour blobs (shared across all vision tasks).
- **Middle layers** detect textures and object parts.
- **Late layers** are task-specific.

By keeping early/middle layers frozen we:
- Avoid destroying already-good representations with noisy gradients.
- Greatly reduce the number of parameters that need training from scratch.

We unfreeze only the last feature block and replace the classification head.

## 5. Model Definitions

In [ ]:
def build_mobilenet_v2(num_labels: int) -> nn.Module:
    """MobileNetV2 — freeze all layers except last 2 feature blocks + new head."""
    m = tv_models.mobilenet_v2(weights=tv_models.MobileNet_V2_Weights.IMAGENET1K_V1)
    for p in m.parameters():
        p.requires_grad = False
    for block in list(m.features.children())[-2:]:
        for p in block.parameters():
            p.requires_grad = True
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(m.classifier[1].in_features, num_labels),
    )
    for p in m.classifier.parameters():
        p.requires_grad = True
    return m


def build_efficientnet_b0(num_labels: int) -> nn.Module:
    """EfficientNet-B0 — freeze all except last feature block + new head."""
    m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for p in m.parameters():
        p.requires_grad = False
    for block in list(m.features.children())[-1:]:
        for p in block.parameters():
            p.requires_grad = True
    in_feat = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_feat, num_labels),
    )
    for p in m.classifier.parameters():
        p.requires_grad = True
    return m


ARCH_FNS = {
    "mobilenet_v2":    build_mobilenet_v2,
    "efficientnet_b0": build_efficientnet_b0,
}

print(f"{'Arch':<22} {'Trainable':>10} {'Total':>12} {'Frozen%':>9}  Size")
print("-" * 70)
for arch, fn in ARCH_FNS.items():
    m         = fn(NUM_LABELS)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in m.parameters())
    frozen_pct = 100.0 * (total - trainable) / total
    size_mb   = total * 4 / 1024 / 1024
    print(f"{arch:<22} {trainable:>10,} {total:>12,} {frozen_pct:>8.1f}%  {size_mb:.1f} MB")

## 6. Train All Lightweight Models

In [ ]:
LR           = 3e-4
WEIGHT_DECAY = 1e-3
MAX_EPOCHS   = 25

all_histories = {}
best_models   = {}
best_val_f1s  = {}
checkpoints   = {}

for arch, fn in ARCH_FNS.items():
    print(f"\n{'='*55}\n  Training: {arch}\n{'='*55}")
    ckpt = CHECKPOINT_DIR / f"{arch}_finetune.pth"
    checkpoints[arch] = ckpt

    best_state, best_f1, history, epochs_run = train_model(
        fn, NUM_LABELS, train_loader, val_loader, DEVICE,
        lr=LR, weight_decay=WEIGHT_DECAY, max_epochs=MAX_EPOCHS, warmup_epochs=3,
    )
    save_checkpoint(best_state, ckpt)

    m = fn(NUM_LABELS).to(DEVICE)
    m.load_state_dict(best_state)
    m.eval()

    all_histories[arch] = history
    best_models[arch]   = m
    best_val_f1s[arch]  = best_f1
    print(f"  Best val F1: {best_f1:.4f}")

print("\n=== Val F1 Summary ===")
for arch in ARCH_FNS:
    print(f"  {arch:<22}  {best_val_f1s[arch]:.4f}")

## 7. Training Curves Comparison

In [ ]:
plot_multi_arch_histories(all_histories, experiment_name="Lightweight Transfer Learning")

## 8. Test-Set Metrics

In [ ]:
print(f"{'Model':<22} {'loss':>7} {'exact':>7} {'hamming':>8} {'IoU':>7} {'prec':>7} {'rec':>7} {'F1':>7}")
print("-" * 85)
for arch, fn in ARCH_FNS.items():
    m = load_checkpoint(fn, NUM_LABELS, checkpoints[arch], DEVICE)
    imgs, lbls, preds, probs = collect_test_predictions(m, test_loader, DEVICE)
    all_logits = []
    m.eval()
    with torch.no_grad():
        for images, _ in test_loader:
            all_logits.append(m(images.to(DEVICE)).cpu())
    logits = torch.cat(all_logits)
    met = compute_multilabel_metrics(lbls, preds, logits=logits)
    print(f"{arch:<22} {met['loss']:>7.4f} {met['exact_match']:>7.4f} "
          f"{met['hamming_acc']:>8.4f} {met['mean_iou']:>7.4f} "
          f"{met['precision_micro']:>7.4f} {met['recall_micro']:>7.4f} "
          f"{met['f1_micro']:>7.4f}")

## 9. Analysis — Best Lightweight Model

In [ ]:
# Pick the architecture with the higher val F1 for detailed analysis
best_arch = max(best_val_f1s, key=best_val_f1s.get)
print(f"Best architecture: {best_arch}  (val F1 = {best_val_f1s[best_arch]:.4f})")

model = load_checkpoint(ARCH_FNS[best_arch], NUM_LABELS, checkpoints[best_arch], DEVICE)
images, labels, preds, probs = collect_test_predictions(model, test_loader, DEVICE)
correct_idx, partial_idx, incorrect_idx = categorize_predictions(labels, preds)

show_prediction_examples(correct_idx,   images, labels, preds, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   images, labels, preds, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, images, labels, preds, "Fully Incorrect",   n=4)

In [ ]:
plot_per_class_metrics(labels, preds)
plot_confusion_matrices(labels, preds)

In [ ]:
show_saliency_examples(correct_idx,   images, labels, preds, model, "Fully Correct",   n=4, device=DEVICE)
show_saliency_examples(incorrect_idx, images, labels, preds, model, "Fully Incorrect", n=4, device=DEVICE)

## 10. Conclusions

- **Transfer learning is a game-changer**: both models dramatically outperform
  all from-scratch networks despite having far fewer trainable parameters.
- The pretrained backbone supplies generalizable visual features that would
  require orders of magnitude more data to learn from scratch.
- EfficientNet-B0 and MobileNetV2 achieve strong F1 with low computational cost.

**Next:** We scale up to more powerful modern backbones — ResNet-50,
EfficientNetV2, and ConvNeXt — with full two-stage fine-tuning (Notebook 5).